# Milestone 3 — Static batching

The per-step floor is paid per **step**, not per token. Batching splits it.

In [ ]:
!git clone https://github.com/Jayaprakash-030/tiny-inference-engine.git
%cd tiny-inference-engine
!pip install -q -e .

In [ ]:
from engine import load
from engine.batched import batched_generate, check_batch_matches_single, sweep
from engine.prompts import BATCH
from engine.results import save

rt = load()
rt.describe()

### Correctness first

In [ ]:
assert check_batch_matches_single(rt, BATCH[:4]), "batching diverged — check padding / position_ids"

In [ ]:
batched_generate(rt, BATCH[:2], max_new_tokens=16)   # warmup
print("warmup done")

### Throughput sweep

In [ ]:
m3 = sweep(rt, sizes=(1, 2, 4, 8, 16, 32), max_new_tokens=200)
save(m3)

In [ ]:
import matplotlib.pyplot as plt

sizes = [r["batch_size"] for r in m3]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, key, title, color in [
    (axes[0], "tokens_per_sec", "Throughput (total tok/s)", "tab:blue"),
    (axes[1], "per_seq_tok_per_sec", "Per-sequence speed (tok/s)", "tab:orange"),
    (axes[2], "peak_mem_gb", "Peak memory (GB)", "tab:green"),
]:
    ax.plot(sizes, [r[key] for r in m3], marker="o", color=color)
    ax.set_title(title)
    ax.set_xlabel("batch size")
    ax.set_xscale("log", base=2)
    ax.set_xticks(sizes)
    ax.set_xticklabels(sizes)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("benchmarks/plots/m3_batching.png", dpi=140)
plt.show()

Throughput climbs then flattens — the knee is where the GPU stops being launch-bound.
Per-sequence speed drifts *down*: throughput up, individual latency down, the central
trade in serving.

`wasted_slot_pct` is the argument for milestone 4.